# Simplified AWG Geometry Notebook

这个 notebook 用于根据当前已经得到的 AWG 初始设计参数，整理一个 `1-input / 4-output / 16-arm` 的 simplified AWG 几何起点。

目标不是直接做高精度仿真，而是先把第一版几何参数、端口位置和 arm length 分配整理清楚，方便后续继续建模。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd()

awg_params_csv = PROJECT_ROOT / "awg_initial_parameters.csv"
awg_channels_csv = PROJECT_ROOT / "awg_channel_plan.csv"
awg_lengths_csv = PROJECT_ROOT / "awg_arrayed_waveguide_lengths.csv"

ports_csv = PROJECT_ROOT / "simplified_awg_geometry_ports.csv"
arms_csv = PROJECT_ROOT / "simplified_awg_geometry_arms.csv"
layout_png = PROJECT_ROOT / "simplified_awg_geometry_layout.png"

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
params_df = pd.read_csv(awg_params_csv)
channels_df = pd.read_csv(awg_channels_csv)
lengths_df = pd.read_csv(awg_lengths_csv)

params_df

In [ ]:
def get_param(name):
    row = params_df.loc[params_df["parameter"] == name, "value"]
    if row.empty:
        raise KeyError(name)
    return float(row.iloc[0])


selected_width_um = get_param("selected_waveguide_width")
waveguide_height_um = get_param("waveguide_height")
delta_length_um = get_param("delta_length")
array_count = int(round(get_param("arrayed_waveguide_count")))
fpr_length_um = get_param("suggested_fpr_length")
array_pitch_um = get_param("suggested_array_pitch")
output_pitch_um = get_param("suggested_output_pitch")
bend_radius_um = get_param("minimum_bend_radius")

print(f"Selected width = {selected_width_um:.2f} um")
print(f"Waveguide height = {waveguide_height_um:.2f} um")
print(f"Delta length = {delta_length_um:.4f} um")

In [ ]:
# First-pass layout assumptions for a simplified geometry.
# These are heuristic starting values, not optimized AWG dimensions.
slab_gap_um = 40.0
input_straight_um = 20.0
output_straight_um = 20.0
array_run_extra_um = 1200.0

# Define four vertical interfaces for a simple 2-FPR sketch.
x_input_port = 0.0
x_input_slab = input_straight_um
x_array_input = x_input_slab + fpr_length_um
x_array_output = x_array_input + slab_gap_um
x_output_slab = x_array_output + fpr_length_um
x_output_port = x_output_slab + output_straight_um

print(x_input_port, x_input_slab, x_array_input, x_array_output, x_output_slab, x_output_port)

In [ ]:
# Compute y positions for array ports and output ports.
array_indices = np.arange(array_count)
array_center = (array_count - 1) / 2
array_y_um = (array_indices - array_center) * array_pitch_um

output_count = len(channels_df)
output_indices = np.arange(output_count)
output_center = (output_count - 1) / 2
output_y_um = (output_indices - output_center) * output_pitch_um

input_y_um = 0.0

print(array_y_um[:5], '...')
print(output_y_um)

In [ ]:
# Build a table for input, array, and output port locations.
rows = []

rows.append({"port_type": "input", "port_id": "in_1", "x_um": x_input_port, "y_um": input_y_um})

for i, y in enumerate(array_y_um, start=1):
    rows.append({"port_type": "array_in", "port_id": f"ain_{i:02d}", "x_um": x_array_input, "y_um": float(y)})
    rows.append({"port_type": "array_out", "port_id": f"aout_{i:02d}", "x_um": x_array_output, "y_um": float(y)})

for i, y in enumerate(output_y_um, start=1):
    rows.append({"port_type": "output", "port_id": f"out_{i}", "x_um": x_output_port, "y_um": float(y)})

ports_df = pd.DataFrame(rows)
ports_df.head()

In [ ]:
# Build a first-pass arm table.
# The absolute base length is heuristic; the important physically derived quantity is delta_length.
base_arm_length_um = 1500.0
arm_total_length_um = base_arm_length_um + lengths_df["relative_length_um"].to_numpy()

arms_df = pd.DataFrame(
    {
        "arm_index": lengths_df["arm_index"],
        "y_um": array_y_um,
        "relative_length_um": lengths_df["relative_length_um"],
        "total_length_um": arm_total_length_um,
        "delta_length_step_um": delta_length_um,
    }
)
arms_df.head()

In [ ]:
# Save reusable geometry tables.
ports_df.to_csv(ports_csv, index=False)
arms_df.to_csv(arms_csv, index=False)

print(f"Saved: {ports_csv}")
print(f"Saved: {arms_csv}")

In [ ]:
# Plot a simplified AWG layout sketch.
plt.figure(figsize=(10, 5))

for x in [x_input_slab, x_array_input, x_array_output, x_output_slab]:
    plt.axvline(x, color="lightgray", linestyle="--", linewidth=1)

plt.scatter([x_input_port], [input_y_um], s=90, label="Input port")
plt.scatter(np.full_like(array_y_um, x_array_input), array_y_um, s=30, label="Array aperture in")
plt.scatter(np.full_like(array_y_um, x_array_output), array_y_um, s=30, label="Array aperture out")
plt.scatter(np.full_like(output_y_um, x_output_port), output_y_um, s=90, label="Output ports")

for y in array_y_um:
    plt.plot([x_input_slab, x_array_input], [0.0, y], color="tab:blue", alpha=0.18)
    plt.plot([x_array_output, x_output_slab], [y, 0.0], color="tab:orange", alpha=0.10)

for y in output_y_um:
    plt.plot([x_output_slab, x_output_port], [y, y], color="tab:green", alpha=0.45)

plt.text(x_input_port, input_y_um + 1.2, "Input", ha="center")
for i, y in enumerate(output_y_um, start=1):
    plt.text(x_output_port + 2, y, f"Out {i}", va="center")

plt.title("Simplified 1x4 AWG Geometry Sketch")
plt.xlabel("x (um)")
plt.ylabel("y (um)")
plt.grid(True, alpha=0.25)
plt.axis("equal")
plt.legend(loc="upper left")
plt.tight_layout()
plt.savefig(layout_png, dpi=200)
plt.show()

print(f"Saved: {layout_png}")

In [ ]:
# Summarize the first-pass geometry values.
summary_df = pd.DataFrame(
    {
        "item": [
            "waveguide_width_um",
            "waveguide_height_um",
            "array_count",
            "output_count",
            "fpr_length_um",
            "slab_gap_um",
            "array_pitch_um",
            "output_pitch_um",
            "bend_radius_um",
            "delta_length_um",
            "base_arm_length_um",
            "max_total_arm_length_um",
            "layout_total_length_um",
            "array_aperture_span_um",
            "output_aperture_span_um",
        ],
        "value": [
            selected_width_um,
            waveguide_height_um,
            array_count,
            output_count,
            fpr_length_um,
            slab_gap_um,
            array_pitch_um,
            output_pitch_um,
            bend_radius_um,
            delta_length_um,
            base_arm_length_um,
            float(arms_df["total_length_um"].max()),
            x_output_port - x_input_port,
            float(array_y_um.max() - array_y_um.min()),
            float(output_y_um.max() - output_y_um.min()),
        ],
    }
)
summary_df

In [ ]:
# Optional views of the generated tables.
ports_df

In [ ]:
arms_df